In [ ]:
import pandas as pd

# House Building Starts

## UK
Source: Indicators of House Building: https://www.ons.gov.uk/peoplepopulationandcommunity/housing/datasets/ukhousebuildingpermanentdwellingsstartedandcompleted

Table: 1b, 1c, 1d

Region: GB

Key Column: Started - Private Enterprises

In [ ]:
with pd.ExcelFile("../data/raw/starts/indicatorsofukhousebuilding.xlsx") as xls:
    eng = pd.read_excel(xls, sheet_name='1b', skiprows=5, index_col=1)
    wal = pd.read_excel(xls, sheet_name='1c', skiprows=5, index_col=1)
    sco = pd.read_excel(xls, sheet_name='1d', skiprows=5, index_col=1)

df_list = [eng, wal, sco]

for df in df_list:
    df.drop("Revised", axis=1, inplace=True)

# Cropping out wales dates
wal = wal.iloc[15:]

df_combined = pd.concat({
    'England' : eng['Started - Private Enterprise'],
    'Wales' : wal['Started - Private Enterprise'],
    'Scotland' : sco['Started - Private Enterprise']}, axis=1)

# Applies the numeric conversion for Wales strings
df_combined = df_combined.apply(pd.to_numeric, errors='coerce')

df_combined['SUM'] = df_combined.sum(axis=1)

# Mapping quarters
quarter_map = {
    "Jan - Mar": "Q1",
    "Apr - Jun": "Q2",
    "Jul - Sep": "Q3",
    "Oct - Dec": "Q4"
}

# Clean up index
idx_str = df_combined.index.astype(str).str.strip()

# Extract the month range text and the 4-digit year
months = idx_str.str.extract(r'(Jan - Mar|Apr - Jun|Jul - Sep|Oct - Dec)')[0]
years = idx_str.str.extract(r'(\d{4})')[0]

# Translate the months into Q1, Q2, etc.
quarters = months.map(quarter_map)

# Glue into strings
quarter_strings = years + quarters

# Convert to Period objects
df_combined.index = pd.PeriodIndex(quarter_strings, freq='Q')

# Naming index
df_combined.index.name = 'Quarter'

# Adding MEEN data
meen_sums = [
    28800, 41600, 41400, 37400,
    37200, 46500, 42800, 28100,
    26200, 38100, 39100, 31400
]

# Generate quarter indices
meen_quarters = pd.period_range(start='1975Q1', periods=len(meen_sums), freq='Q')

# Creating new df with jut sum col
df_meen = pd.DataFrame({'SUM': meen_sums}, index=meen_quarters)
df_meen.index.name = 'Quarter'

# Concat with combined
df_combined = pd.concat([df_meen, df_combined], axis=0)
# Saving to csv
df_combined.to_csv('../data/python_master/starts/UK_starts_series.csv', index=True)

# England
We scale the rows from 1975Q1 - 1977Q4 by 0.825 then just us the England series

In [ ]:
df_eng = pd.read_csv('../data/python_master/starts/UK_starts_series.csv', index_col=0, usecols=[0,1,2])

# Scaling by 0.846
df_eng.loc[:'1977Q4', 'SUM'] = df_eng.loc[:'1977Q4', 'SUM'] * 0.846

# Filling in values from SUM
df_eng.loc[:'1977Q4', 'England'] = df_eng.loc[:'1977Q4', 'SUM']

df_eng.to_csv('../data/python_master/starts/ENG_starts_series.csv', columns=['England'], index=True)

# Nominal House Prices
# UK
Source: UK House Price Index: data downloads March 2026 (Average Price) - extract UK data
https://www.gov.uk/government/statistical-data-sets/uk-house-price-index-data-downloads-march-2026

In [ ]:
df = pd.read_csv('../data/raw/nominal_house_price/Average-prices-2026-03.csv', index_col=0, parse_dates=[0])

df.drop(columns=['Area_Code', 'Monthly_Change', 'Annual_Change', 'Average_Price_SA'], inplace=True)

In [ ]:
# Filtering to UK
df_uk = df[df['Region_Name'] == 'United Kingdom']

# Filtering date from Jan 1975
df_uk = df_uk.loc['1975-01-01':]

# Taking sums of months for quarters
df_uk = df_uk.resample('QE').mean(numeric_only=True)

# Adding Region Name back
df_uk['Region_Name'] = 'United Kingdom'

# Move to front
df_uk = df_uk[['Region_Name'] + [col for col in df_uk.columns if col != 'Region_Name']]

# Converting to periods
df_uk.index = df_uk.index.to_period('Q')

df_uk.to_csv("../data/python_master/house_price/UK_house_price.csv")

# England
Same exact method but filter for England

In [ ]:
# Filtering to UK
df_eng = df[df['Region_Name'] == 'England']

# Filtering date from Jan 1975
df_eng = df_eng.loc['1975-01-01':]

# Taking sums of months for quarters
df_eng = df_eng.resample('QE').mean(numeric_only=True)

# Adding Region Name back
df_eng['Region_Name'] = 'England'

# Move to front
df_eng = df_eng[['Region_Name'] + [col for col in df_eng.columns if col != 'Region_Name']]

# Converting to periods
df_eng.index = df_eng.index.to_period('Q')

df_eng.to_csv("../data/python_master/house_price/ENG_house_price.csv")

In [ ]:
df_eng

# Construction Costs
## Both UK and England
SPLICE

1st Series:
BIS quarterly construction price and cost indices: July to September 2014 - Output price indices July to September 2014

https://www.gov.uk/government/statistics/bis-quarterly-construction-price-and-cost-indices-july-to-september-2014

Table 1 Tab

Key Variable: Private Housing


2nd Series:
ONS Construction output price indices

https://www.ons.gov.uk/businessindustryandtrade/constructionindustry/datasets/interimconstructionoutputpriceindices

New work tab

Key Variable: Housing (public and private)  index 2015=100


## IMPORTANT: Rebasing of second series is necessary

In [152]:
bis = pd.read_excel('../data/raw/construction_cost/14-p157e-output-price-indices-2014Q2_rev1.xls',
                    sheet_name='Table 1',
                    skiprows=148,
                    usecols=[1,2,5],)

# Cropping comments
bis = bis.iloc[:-6]

bis.columns = ['Year', 'Quarter', 'Output Price']

# Forward fill missing years
bis['Year'] = bis['Year'].ffill()

bis.index = pd.PeriodIndex([f"{y}Q{q}" for y, q in zip(bis['Year'], bis['Quarter'].astype(str).str.replace('Q', ''))], freq='Q')

bis = bis.drop(columns=['Year', 'Quarter'])

In [175]:
ons = pd.read_excel('../data/raw/construction_cost/bulletindataset9.xlsx',
                    sheet_name='New work',
                    skiprows=4,
                    usecols=[0,1],)

# Extract the Year and Month into separate working columns
# If the row contains a 4-digit number (like 2014), extract it as the year
ons['Year'] = ons['Time period'].str.extract(r'(\d{4})')
ons['Month'] = ons['Time period'].str.replace(r'\d{4}\s*', '', regex=True)

# 2. Forward-fill the Year column down the empty rows
ons['Year'] = ons['Year'].ffill()

# 3. Combine Year and Month into a standard string format (e.g., "2014 Jan")
ons['Date_String'] = ons['Year'] + ' ' + ons['Month']

# 4. Convert that string into a true pandas Datetime index
ons.index = pd.to_datetime(ons['Date_String'], format='%Y %b')

# 5. Clean up the DataFrame by dropping the temporary columns
ons = ons.drop(columns=['Time period', 'Year', 'Month', 'Date_String'])

ons.columns =['Output_Index']
# Calculating rolling average
ons['3_Month_Rolling_Avg'] = ons['Output_Index'].rolling(window=3).mean()

In [176]:
ons.head(20)

,Output_Index,3_Month_Rolling_Avg
Date_String,,
2014-01-01,100.5,NaN
2014-02-01,99.8,NaN
2014-03-01,99.3,99.866667
2014-04-01,98.8,99.300000
2014-05-01,98.4,98.833333
2014-06-01,99.1,98.766667
2014-07-01,99.4,98.966667
2014-08-01,98.8,99.100000
2014-09-01,99.1,99.100000
